# ARC v0.20 — HNSW Frozen Mechanism Replication

This notebook implements a graph-based ANN boundary replication for the current approximation-under-feedback study.

**Scientific rule:** the HNSW low/high `efSearch` contrast must be chosen from FIT one-shot quality/cost evidence only. Do not inspect held-out feedback-trajectory endpoints before freezing the contrast.


## Drive-integrated lineage

This notebook is wired to the existing frozen ARC artifacts in Google Drive rather than requiring re-encoding.

Verified lineage:

- ARC-v0.18 run root: `/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/cross-encoder-fever-replication-v018/20260819-015645`
- corpus embedding manifest: `corpus_encoding_manifest.json`
- corpus embeddings: 55 `float16` shards, total 5,416,568 rows
- DEV query embeddings: `dev_query_embeddings.float32.npy`
- DEV query IDs: `dev_query_ids.txt`
- authoritative FEVER FIT/validation split: ARC-v0.13 `v013_boundary_query_split.csv`
- FEVER DEV qrels: `/content/drive/MyDrive/rag-pq-checkpoints/raw-datasets/fever/qrels/dev.tsv`

The v0.18 protocol fixes E5-small-v2, 384 dimensions, `query:` / `passage:` prefixes, the 44-policy grid, 4 feedback updates, top-100 retrieval, nDCG@10 utility, and epsilon=0.002.

This notebook must not access FEVER test qrels.


In [ ]:

# Colab/Drive-integrated paths
from pathlib import Path
import json, hashlib, os, numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DRIVE_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints")
ARC_ROOT = DRIVE_ROOT / "arc-v0"

V018_RUN = ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-015645"
V018_SHARDS = V018_RUN / "corpus_shards"
V018_QUERY_EMB = V018_RUN / "dev_query_embeddings.float32.npy"
V018_QUERY_IDS = V018_RUN / "dev_query_ids.txt"
V018_MANIFEST = V018_RUN / "corpus_encoding_manifest.json"
V018_PROTOCOL = V018_RUN / "v018_cross_encoder_protocol.json"

V013_RUN = ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-151852"
V013_SPLIT = V013_RUN / "v013_boundary_query_split.csv"

FEVER_QRELS_DEV = DRIVE_ROOT / "raw-datasets" / "fever" / "qrels" / "dev.tsv"

PATHS = {
    "v018_run": V018_RUN,
    "v018_shards": V018_SHARDS,
    "query_embeddings": V018_QUERY_EMB,
    "query_ids": V018_QUERY_IDS,
    "corpus_manifest": V018_MANIFEST,
    "v018_protocol": V018_PROTOCOL,
    "v013_split": V013_SPLIT,
    "fever_dev_qrels": FEVER_QRELS_DEV,
}

for name, p in PATHS.items():
    print(f"{name:20s}", "OK" if p.exists() else "MISSING", p)


## Frozen-lineage integrity audit

In [ ]:

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def load_and_audit_lineage():
    protocol = json.loads(V018_PROTOCOL.read_text())
    manifest = json.loads(V018_MANIFEST.read_text())

    assert protocol["encoder"] == "intfloat/e5-small-v2"
    assert protocol["dimension"] == 384
    assert protocol["feedback"]["rounds"] == 4
    assert protocol["feedback"]["top_retrieve"] == 100
    assert protocol["feedback"]["utility_k"] == 10
    assert float(protocol["epsilon_primary"]) == 0.002

    assert manifest["status"] == "COMPLETE"
    assert int(manifest["rows"]) == 5_416_568
    assert len(manifest["shards"]) == 55
    assert sum(int(s["rows"]) for s in manifest["shards"]) == 5_416_568

    expected = list(range(55))
    observed = [int(s["shard"]) for s in manifest["shards"]]
    assert observed == expected, (observed[:5], observed[-5:])

    # Verify local shard presence and hashes before building HNSW.
    bad = []
    for s in manifest["shards"]:
        sid = int(s["shard"])
        emb = V018_SHARDS / f"shard-{sid:04d}.float16.npy"
        ids = V018_SHARDS / f"shard-{sid:04d}.ids.txt"
        if not emb.exists() or not ids.exists():
            bad.append((sid, "missing"))
            continue
        got = sha256_file(emb)
        if got != s["sha256"]:
            bad.append((sid, f"embedding sha mismatch {got} != {s['sha256']}"))
    if bad:
        raise RuntimeError(f"Corpus lineage audit failed: {bad[:10]}")

    print("Lineage audit PASS")
    print("Rows:", manifest["rows"])
    print("Shards:", len(manifest["shards"]))
    print("FIT SHA256:", protocol["fit_membership_sha256"])
    print("VAL SHA256:", protocol["validation_membership_sha256"])
    return protocol, manifest

# protocol, corpus_manifest = load_and_audit_lineage()


## Load the existing E5 FEVER artifacts

In [ ]:

def read_id_lines(path):
    return [x.rstrip("\n") for x in path.open("r", encoding="utf-8")]

def load_dev_queries():
    q = np.load(V018_QUERY_EMB, mmap_mode="r")
    qids = read_id_lines(V018_QUERY_IDS)
    assert q.shape == (6666, 384), q.shape
    assert len(qids) == 6666
    norms = np.linalg.norm(np.asarray(q[:512], dtype=np.float32), axis=1)
    assert np.allclose(norms, 1.0, atol=2e-3)
    return q, np.asarray(qids, dtype=object)

def load_split():
    s = pd.read_csv(V013_SPLIT)
    # Robust column discovery: expected columns include query_id and split/membership.
    print("split columns:", list(s.columns))
    qcol = next(c for c in s.columns if c.lower() in {"query_id","qid","queryid"})
    scol = next(c for c in s.columns if c.lower() in {"split","membership","subset","role"})
    s[qcol] = s[qcol].astype(str)
    s[scol] = s[scol].astype(str).str.lower()

    fit = set(s.loc[s[scol].str.contains("fit"), qcol])
    val = set(s.loc[s[scol].str.contains("val"), qcol])

    assert len(fit) == 3350, len(fit)
    assert len(val) == 3316, len(val)
    assert fit.isdisjoint(val)
    return fit, val

def load_qrels_dev():
    # BEIR qrels TSV is expected as: query-id, corpus-id, score
    q = pd.read_csv(FEVER_QRELS_DEV, sep="\t")
    # Support either header naming convention.
    ren = {}
    for c in q.columns:
        lc = c.lower().replace("_","-")
        if lc in {"query-id","queryid","qid"}:
            ren[c] = "query_id"
        elif lc in {"corpus-id","doc-id","docid","corpusid"}:
            ren[c] = "doc_id"
        elif lc in {"score","relevance","rel"}:
            ren[c] = "relevance"
    q = q.rename(columns=ren)
    if not {"query_id","doc_id","relevance"} <= set(q.columns):
        # Headerless fallback.
        q = pd.read_csv(FEVER_QRELS_DEV, sep="\t", names=["query_id","doc_id","relevance"])
    q["query_id"] = q["query_id"].astype(str)
    q["doc_id"] = q["doc_id"].astype(str)
    q["relevance"] = q["relevance"].astype(float)
    return q

def iter_corpus_shards(manifest):
    for s in manifest["shards"]:
        sid = int(s["shard"])
        embp = V018_SHARDS / f"shard-{sid:04d}.float16.npy"
        idsp = V018_SHARDS / f"shard-{sid:04d}.ids.txt"
        x = np.load(embp, mmap_mode="r")
        ids = read_id_lines(idsp)
        assert x.shape == (int(s["rows"]), 384), (sid, x.shape, s["rows"])
        assert len(ids) == len(x)
        yield sid, x, ids


## Build HNSW directly from the 55 frozen v0.18 embedding shards

In [ ]:

HNSW_OUT = ARC_ROOT / "hnsw-mechanism-replication-v020"
HNSW_OUT.mkdir(parents=True, exist_ok=True)

INDEX_PATH = HNSW_OUT / "fever-e5-small-v2-hnsw-m32-efc200.faiss"
DOC_IDS_PATH = HNSW_OUT / "fever-e5-doc-ids.txt"

def build_hnsw_from_v018_shards(manifest):
    index = faiss.IndexHNSWFlat(
        384,
        CONFIG["M"],
        faiss.METRIC_INNER_PRODUCT,
    )
    index.hnsw.efConstruction = CONFIG["efConstruction"]
    index.verbose = True

    all_ids = []
    total = 0
    for sid, x16, ids in iter_corpus_shards(manifest):
        x = np.asarray(x16, dtype=np.float32)
        # semantic audit before add
        assert np.isfinite(x).all()
        norms = np.linalg.norm(x[:min(2048, len(x))], axis=1)
        if not np.allclose(norms, 1.0, atol=3e-3):
            raise ValueError(f"shard {sid}: normalization audit failed")
        index.add(x)
        all_ids.extend(ids)
        total += len(x)
        print(f"added shard {sid:02d}: {total:,}/5,416,568")

    assert index.ntotal == 5_416_568
    assert len(all_ids) == 5_416_568

    faiss.write_index(index, str(INDEX_PATH))
    DOC_IDS_PATH.write_text("\n".join(all_ids) + "\n", encoding="utf-8")

    build_record = {
        "index_type": "IndexHNSWFlat",
        "metric": "inner_product",
        "M": CONFIG["M"],
        "efConstruction": CONFIG["efConstruction"],
        "ntotal": int(index.ntotal),
        "source_v018_manifest_sha256": sha256_file(V018_MANIFEST),
        "source_v018_protocol_sha256": sha256_file(V018_PROTOCOL),
        "index_sha256": sha256_file(INDEX_PATH),
        "doc_ids_sha256": sha256_file(DOC_IDS_PATH),
    }
    (HNSW_OUT/"hnsw_build_record.json").write_text(json.dumps(build_record, indent=2))
    return index, np.asarray(all_ids, dtype=object)

# protocol, corpus_manifest = load_and_audit_lineage()
# index, doc_ids = build_hnsw_from_v018_shards(corpus_manifest)


## Verify the v0.13 FIT/validation membership against the v0.18 protocol

In [ ]:

def canonical_membership_hash(ids):
    # Match the paper's membership-level reproducibility contract:
    # sorted unique query IDs, newline terminated.
    payload = ("\n".join(sorted(map(str, ids))) + "\n").encode()
    return hashlib.sha256(payload).hexdigest()

def audit_split_against_v018_protocol(protocol, fit_ids, val_ids):
    got_fit = canonical_membership_hash(fit_ids)
    got_val = canonical_membership_hash(val_ids)
    print("FIT:", got_fit, "expected:", protocol["fit_membership_sha256"])
    print("VAL:", got_val, "expected:", protocol["validation_membership_sha256"])
    assert got_fit == protocol["fit_membership_sha256"]
    assert got_val == protocol["validation_membership_sha256"]
    print("Split lineage PASS")

# fit_ids, val_ids = load_split()
# audit_split_against_v018_protocol(protocol, fit_ids, val_ids)


## FIT-only HNSW one-shot calibration

In [ ]:

def run_fit_one_shot_ladder(index, queries, query_ids, doc_ids, qrels, fit_ids):
    qrel_map = build_qrel_map(qrels)
    mask = np.array([str(qid) in fit_ids for qid in query_ids])
    q = np.asarray(queries[mask], dtype=np.float32)
    qids = np.asarray(query_ids[mask])

    rows = []
    for ef in CONFIG["ef_ladder"]:
        index.hnsw.efSearch = int(ef)
        t0 = time.perf_counter()
        scores, I = index.search(q, CONFIG["top_k"])
        elapsed = time.perf_counter() - t0

        vals = []
        for i, qid in enumerate(qids):
            ids = [doc_ids[j] for j in I[i] if j >= 0]
            vals.append(ndcg_at_k(ids, qrel_map.get(str(qid), {}), 10))

        rows.append({
            "efSearch": ef,
            "mean_ndcg10": float(np.mean(vals)),
            "latency_ms_per_query": elapsed * 1000 / len(q),
            "n_queries": len(q),
        })
        print(rows[-1])

    df = pd.DataFrame(rows)
    df.to_csv(HNSW_OUT/"v020_fit_one_shot_hnsw_ladder.csv", index=False)
    return df

# queries, query_ids = load_dev_queries()
# qrels = load_qrels_dev()
# doc_ids = np.asarray(read_id_lines(DOC_IDS_PATH), dtype=object)
# ladder = run_fit_one_shot_ladder(index, queries, query_ids, doc_ids, qrels, fit_ids)
# ladder


In [ ]:

DATA = ROOT / "data"

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

required = [
    DATA/"fever_e5_corpus.npy",
    DATA/"fever_e5_queries.npy",
    DATA/"query_ids.npy",
    DATA/"doc_ids.npy",
    DATA/"fit_query_ids.txt",
    DATA/"validation_query_ids.txt",
    DATA/"qrels.parquet",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing inputs (expected before full execution):")
    for p in missing:
        print(" -", p)
else:
    print("All required inputs found.")


In [ ]:

def load_inputs():
    corpus = np.load(DATA/"fever_e5_corpus.npy", mmap_mode="r")
    queries = np.load(DATA/"fever_e5_queries.npy", mmap_mode="r")
    query_ids = np.load(DATA/"query_ids.npy", allow_pickle=True)
    doc_ids = np.load(DATA/"doc_ids.npy", allow_pickle=True)
    qrels = pd.read_parquet(DATA/"qrels.parquet")
    fit_ids = set((DATA/"fit_query_ids.txt").read_text().splitlines())
    val_ids = set((DATA/"validation_query_ids.txt").read_text().splitlines())

    assert corpus.shape[1] == CONFIG["dim"]
    assert queries.shape[1] == CONFIG["dim"]
    assert corpus.dtype == np.float32
    assert queries.dtype == np.float32
    assert len(query_ids) == len(queries)
    assert len(doc_ids) == len(corpus)
    assert fit_ids.isdisjoint(val_ids)

    # Sample normalization audit.
    for arr, name in [(corpus, "corpus"), (queries, "queries")]:
        idx = np.linspace(0, len(arr)-1, min(1000, len(arr)), dtype=int)
        norms = np.linalg.norm(np.asarray(arr[idx]), axis=1)
        if not np.allclose(norms, 1.0, atol=1e-3):
            raise ValueError(f"{name} embeddings are not L2 normalized.")

    return corpus, queries, query_ids, doc_ids, qrels, fit_ids, val_ids

# Uncomment after inputs are present:
# corpus, queries, query_ids, doc_ids, qrels, fit_ids, val_ids = load_inputs()


## 2. Build / load HNSW index

In [ ]:

INDEX_PATH = ART/"fever_e5_hnsw_m32_efc200.faiss"

def build_hnsw(corpus):
    index = faiss.IndexHNSWFlat(
        CONFIG["dim"],
        CONFIG["M"],
        faiss.METRIC_INNER_PRODUCT,
    )
    index.hnsw.efConstruction = CONFIG["efConstruction"]
    index.verbose = True

    batch = 100_000
    for start in range(0, len(corpus), batch):
        stop = min(start + batch, len(corpus))
        index.add(np.asarray(corpus[start:stop], dtype=np.float32))
        print(f"added {stop:,}/{len(corpus):,}")

    faiss.write_index(index, str(INDEX_PATH))
    return index

def load_hnsw():
    return faiss.read_index(str(INDEX_PATH))

# Example:
# index = build_hnsw(corpus) if not INDEX_PATH.exists() else load_hnsw()


## 3. Retrieval and nDCG@10

In [ ]:

def hnsw_search(index, q, ef_search, k=None):
    if k is None:
        k = CONFIG["top_k"]
    index.hnsw.efSearch = int(ef_search)  # explicit reset before every call
    q = np.asarray(q, dtype=np.float32)
    return index.search(q, k)

def build_qrel_map(qrels):
    out = {}
    for qid, sub in qrels.groupby("query_id"):
        out[str(qid)] = {str(d): float(r) for d, r in zip(sub.doc_id, sub.relevance)}
    return out

def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    if len(rels) == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, len(rels)+2))
    gains = np.power(2.0, rels) - 1.0
    return float(np.sum(gains * discounts))

def ndcg_at_k(retrieved_doc_ids, qrel_dict, k=10):
    rels = [qrel_dict.get(str(d), 0.0) for d in retrieved_doc_ids[:k]]
    ideal = sorted(qrel_dict.values(), reverse=True)[:k]
    denom = dcg(ideal)
    return 0.0 if denom == 0 else dcg(rels) / denom


## 4. One-shot FIT calibration ladder

This stage is allowed to inspect **only** one-shot quality/cost. It must not run any feedback trajectory analysis.

Use the FIT query set. After choosing the low/high contrast, write it to the manifest and stop changing it.


In [ ]:

def query_mask(query_ids, selected_ids):
    selected_ids = {str(x) for x in selected_ids}
    return np.array([str(q) in selected_ids for q in query_ids], dtype=bool)

def run_one_shot_ladder(index, queries, query_ids, doc_ids, qrels, fit_ids):
    qrel_map = build_qrel_map(qrels)
    mask = query_mask(query_ids, fit_ids)
    q = np.asarray(queries[mask], dtype=np.float32)
    qids = np.asarray(query_ids[mask])

    rows = []
    for ef in CONFIG["ef_ladder"]:
        t0 = time.perf_counter()
        scores, I = hnsw_search(index, q, ef_search=ef, k=CONFIG["top_k"])
        dt = time.perf_counter() - t0

        vals = []
        for i, qid in enumerate(qids):
            rid = [doc_ids[j] for j in I[i] if j >= 0]
            vals.append(ndcg_at_k(rid, qrel_map.get(str(qid), {}), 10))

        rows.append({
            "efSearch": ef,
            "mean_ndcg10": float(np.mean(vals)),
            "latency_ms_per_query": 1000.0 * dt / len(q),
            "n_queries": len(q),
        })
        print(rows[-1])

    df = pd.DataFrame(rows)
    df.to_csv(ART/"fit_one_shot_hnsw_ladder.csv", index=False)
    return df

# ladder = run_one_shot_ladder(index, queries, query_ids, doc_ids, qrels, fit_ids)
# ladder


## 5. Freeze the contrast

Fill `LOW_EF` and `HIGH_EF` only after inspecting the FIT one-shot ladder. The rule is to choose a contrast with a meaningful quality/cost difference, **without any trajectory outcome information**.


In [ ]:

LOW_EF = None   # e.g. 16, but do not precommit until FIT one-shot calibration
HIGH_EF = None  # e.g. 128

def freeze_contrast(low_ef, high_ef):
    if low_ef not in CONFIG["ef_ladder"] or high_ef not in CONFIG["ef_ladder"]:
        raise ValueError("efSearch values must come from the prespecified ladder.")
    if low_ef >= high_ef:
        raise ValueError("Expected low_ef < high_ef.")

    freeze = {
        "low_efSearch": int(low_ef),
        "high_efSearch": int(high_ef),
        "selection_basis": "FIT one-shot quality/cost only",
        "seed": CONFIG["seed"],
        "faiss_version": faiss.__version__,
        "index_path": str(INDEX_PATH),
        "index_sha256": sha256_file(INDEX_PATH) if INDEX_PATH.exists() else None,
    }
    (ART/"FROZEN_HNSW_CONTRAST.json").write_text(json.dumps(freeze, indent=2))
    print(json.dumps(freeze, indent=2))
    return freeze

# freeze = freeze_contrast(LOW_EF, HIGH_EF)


## 6. Frozen 44-policy grid

In [ ]:

def make_policies():
    policies = []
    for a in CONFIG["alphas"]:
        for k in CONFIG["mean_k"]:
            policies.append({"family":"mean", "alpha":a, "k":k, "temperature":None})
        for k in CONFIG["softmax_k"]:
            for tau in CONFIG["temperatures"]:
                policies.append({"family":"softmax", "alpha":a, "k":k, "temperature":tau})
    assert len(policies) == 44
    return policies

POLICIES = make_policies()
pd.DataFrame(POLICIES).head(), len(POLICIES)


## 7. Coupled feedback trajectories

In [ ]:

def normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(n, eps)

def feedback_vector(doc_vecs, scores, family, temperature=None):
    doc_vecs = np.asarray(doc_vecs, dtype=np.float32)
    if family == "mean":
        f = doc_vecs.mean(axis=0)
    elif family == "softmax":
        tau = float(temperature)
        z = np.asarray(scores, dtype=np.float64) / tau
        z = z - z.max()
        w = np.exp(z)
        w = w / w.sum()
        f = (doc_vecs * w[:, None]).sum(axis=0)
    else:
        raise ValueError(f"Unknown family: {family}")
    return normalize_rows(f[None, :])[0]

def update_state(q0, f, alpha):
    x = (1.0 - alpha) * q0 + alpha * f
    return normalize_rows(x[None, :])[0]

def jaccard_distance(a, b):
    A, B = set(map(int, a)), set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ols_slope(y):
    y = np.asarray(y, dtype=float)
    x = np.arange(len(y), dtype=float)
    return float(np.polyfit(x, y, 1)[0])

def single_trajectory_pair(index, corpus, q0, qid, doc_ids, qrel_map, policy, low_ef, high_ef):
    # Both branches begin from identical q0.
    qL = np.asarray(q0, dtype=np.float32).copy()
    qH = np.asarray(q0, dtype=np.float32).copy()

    d_hist, j_hist, abs_hist, signed_hist = [], [], [], []
    uL_hist, uH_hist = [], []

    for t in range(CONFIG["rounds"] + 1):
        # Important: explicitly set efSearch before every branch search.
        sL, iL = hnsw_search(index, qL[None, :], low_ef, CONFIG["top_k"])
        sH, iH = hnsw_search(index, qH[None, :], high_ef, CONFIG["top_k"])
        sL, iL = sL[0], iL[0]
        sH, iH = sH[0], iH[0]

        docsL = [doc_ids[j] for j in iL if j >= 0]
        docsH = [doc_ids[j] for j in iH if j >= 0]
        qrels_q = qrel_map.get(str(qid), {})

        uL = ndcg_at_k(docsL, qrels_q, 10)
        uH = ndcg_at_k(docsH, qrels_q, 10)

        d_hist.append(1.0 - float(np.dot(qL, qH)))
        j_hist.append(jaccard_distance(iL[iL>=0], iH[iH>=0]))
        abs_hist.append(abs(uH - uL))
        signed_hist.append(uH - uL)
        uL_hist.append(uL)
        uH_hist.append(uH)

        if t == CONFIG["rounds"]:
            break

        k = int(policy["k"])

        vecL = np.asarray(corpus[iL[:k]], dtype=np.float32)
        vecH = np.asarray(corpus[iH[:k]], dtype=np.float32)

        fL = feedback_vector(vecL, sL[:k], policy["family"], policy["temperature"])
        fH = feedback_vector(vecH, sH[:k], policy["family"], policy["temperature"])

        # Anchored update, matching the paper: (1-alpha) q0 + alpha F_t
        qL = update_state(q0, fL, policy["alpha"])
        qH = update_state(q0, fH, policy["alpha"])

    return {
        "query_id": str(qid),
        **policy,
        "H1": ols_slope(d_hist),
        "H2": ols_slope(j_hist),
        "H3abs": ols_slope(abs_hist),
        "H3signed": ols_slope(signed_hist),
        "final_uL": uL_hist[-1],
        "final_uH": uH_hist[-1],
        "final_signed_gap": uH_hist[-1] - uL_hist[-1],
        "d_hist": d_hist,
        "j_hist": j_hist,
        "abs_hist": abs_hist,
        "signed_hist": signed_hist,
    }


## 8. Run held-out validation

This is the first stage that inspects feedback outcomes. Run it only after the contrast is frozen.


In [ ]:

def run_split(index, corpus, queries, query_ids, doc_ids, qrels, selected_ids, low_ef, high_ef, out_name):
    qrel_map = build_qrel_map(qrels)
    mask = query_mask(query_ids, selected_ids)
    q = np.asarray(queries[mask], dtype=np.float32)
    qids = np.asarray(query_ids[mask])

    rows = []
    total = len(q) * len(POLICIES)
    done = 0

    for qi, (q0, qid) in enumerate(zip(q, qids)):
        for policy in POLICIES:
            rows.append(
                single_trajectory_pair(
                    index, corpus, q0, qid, doc_ids, qrel_map,
                    policy, low_ef, high_ef
                )
            )
            done += 1
        if (qi + 1) % 50 == 0:
            print(f"queries {qi+1}/{len(q)} | events {done}/{total}")

    df = pd.DataFrame(rows)
    df.to_parquet(ART/f"{out_name}.parquet", index=False)
    return df

# Validation only after freeze:
# val_df = run_split(index, corpus, queries, query_ids, doc_ids, qrels,
#                    val_ids, LOW_EF, HIGH_EF, "hnsw_validation_endpoints")


## 9. Query-level aggregate endpoints and query-cluster bootstrap

In [ ]:

def query_means(df, metric):
    return df.groupby("query_id", sort=False)[metric].mean()

def bootstrap_mean_ci(df, metric, reps=None, seed=None):
    reps = reps or CONFIG["bootstrap_reps"]
    seed = CONFIG["seed"] if seed is None else seed
    qrng = np.random.default_rng(seed)

    qids = df["query_id"].drop_duplicates().to_numpy()
    grouped = {qid: df.loc[df.query_id == qid, metric].to_numpy() for qid in qids}

    # First average within query, matching the paper's statistical unit.
    qvals = np.array([grouped[q].mean() for q in qids], dtype=float)
    obs = float(qvals.mean())

    boot = np.empty(reps, dtype=float)
    n = len(qvals)
    for b in range(reps):
        idx = qrng.integers(0, n, size=n)
        boot[b] = qvals[idx].mean()

    lo, hi = np.quantile(boot, [0.025, 0.975])
    return {"mean": obs, "ci_low": float(lo), "ci_high": float(hi), "reps": reps}

def summarize_endpoints(df):
    rows = []
    for m in ["H1", "H2", "H3abs", "H3signed"]:
        r = bootstrap_mean_ci(df, m)
        r["metric"] = m
        rows.append(r)
    return pd.DataFrame(rows)[["metric","mean","ci_low","ci_high","reps"]]

# endpoint_summary = summarize_endpoints(val_df)
# endpoint_summary


## 10. Regime prevalence and signed direction

In [ ]:

def classify_regime(h3, eps):
    if h3 > eps:
        return "amplify"
    if h3 < -eps:
        return "contract"
    return "stable"

def regime_summary(df, eps=0.002):
    x = df.copy()
    x["regime"] = x["H3abs"].map(lambda z: classify_regime(z, eps))
    prevalence = x["regime"].value_counts(normalize=True).rename("fraction").to_frame()
    prevalence["percent"] = 100 * prevalence["fraction"]
    return prevalence

def higher_wins_given_amplification(df, eps=0.002):
    x = df[df["H3abs"] > eps].copy()
    if len(x) == 0:
        return {"n":0, "higher_win":np.nan, "lower_win":np.nan, "tie":np.nan}
    gap = x["final_signed_gap"].to_numpy()
    return {
        "n": int(len(x)),
        "higher_win": float(np.mean(gap > 0)),
        "lower_win": float(np.mean(gap < 0)),
        "tie": float(np.mean(gap == 0)),
    }

def regime_sensitivity(df):
    rows = []
    for eps in CONFIG["epsilon_sensitivity"]:
        r = regime_summary(df, eps)
        rows.append({
            "epsilon": eps,
            "stable_pct": float(r.loc["stable","percent"]) if "stable" in r.index else 0.0,
            "amplify_pct": float(r.loc["amplify","percent"]) if "amplify" in r.index else 0.0,
            "contract_pct": float(r.loc["contract","percent"]) if "contract" in r.index else 0.0,
        })
    return pd.DataFrame(rows)

# regime_summary(val_df, CONFIG["epsilon"])
# higher_wins_given_amplification(val_df, CONFIG["epsilon"])
# regime_sensitivity(val_df)


## 11. Configuration reproducibility: FIT vs validation

In [ ]:

def config_key_cols():
    return ["family", "alpha", "k", "temperature"]

def amplification_by_config(df, eps=0.002):
    x = df.copy()
    x["amplify"] = x["H3abs"] > eps
    return x.groupby(config_key_cols(), dropna=False)["amplify"].mean().reset_index()

def fit_val_correlations(fit_df, val_df, eps=0.002):
    a = amplification_by_config(fit_df, eps).rename(columns={"amplify":"fit"})
    b = amplification_by_config(val_df, eps).rename(columns={"amplify":"val"})
    m = a.merge(b, on=config_key_cols(), how="inner")

    pearson = m["fit"].corr(m["val"], method="pearson")
    spearman = m["fit"].corr(m["val"], method="spearman")

    # Alpha-centered correlation.
    for col in ["fit", "val"]:
        m[col+"_centered"] = m[col] - m.groupby("alpha")[col].transform("mean")
    pearson_centered = m["fit_centered"].corr(m["val_centered"], method="pearson")
    spearman_centered = m["fit_centered"].corr(m["val_centered"], method="spearman")

    return {
        "pearson": float(pearson),
        "spearman": float(spearman),
        "pearson_alpha_centered": float(pearson_centered),
        "spearman_alpha_centered": float(spearman_centered),
    }

# fit_df = run_split(... fit_ids ..., out_name="hnsw_fit_endpoints")
# fit_val_correlations(fit_df, val_df)


## 12. Integrity report

In [ ]:

def write_integrity_report():
    paths = {
        "index": INDEX_PATH,
        "corpus_embeddings": DATA/"fever_e5_corpus.npy",
        "query_embeddings": DATA/"fever_e5_queries.npy",
        "fit_query_ids": DATA/"fit_query_ids.txt",
        "validation_query_ids": DATA/"validation_query_ids.txt",
    }
    report = {}
    for name, p in paths.items():
        report[name] = {
            "path": str(p),
            "exists": p.exists(),
            "sha256": sha256_file(p) if p.exists() else None
        }
    (ART/"integrity_report.json").write_text(json.dumps(report, indent=2))
    return report

# write_integrity_report()


## 13. Interpretation rule

Do not label the experiment a success only if HNSW amplifies.

- `CI(H3abs) > 0`: cross-family transfer of aggregate utility-gap amplification.
- `CI(H3abs) < 0`: mechanism-level reversal / contraction.
- CI contains 0: null/stable boundary.

In every case, report H1/H2/H3abs/H3signed, regime prevalence, epsilon sensitivity, and final signed direction conditional on amplification.
